# Shortest Path Problem

**Objectives**

- Introduce students to the concept of a shortest path tree
- Show students the inner workings of a combinatorial algorithm
- Demonstrate the usefulness of sensitivity analysis in problem solving

**Reading:** Read Handout 3 on the shortest path problem.

**Brief description:** In this lab, we review some motivation and observations behind Dijkstra's algorithm for shortest path computation, and analyze how sensitive the solution to the shortest path problem is to changes in the input data.

 <br>


In [ ]:
# Imports -- don't forget to run this cell!
import vinal as vl
import pandas as pd
from IPython.display import Image
from bokeh.io import output_notebook, show
import numpy as np
import math
import itertools
import networkx as nx
output_notebook()

## Part I: The Shortest Path Problem and Dijkstra's Algorithm

As an entrepreneurial first-year student, you plan to start a pizza delivery company: Good Pizza, Inc.  You will base Good Pizza out of the kitchen on your floor of Jameson Hall and deliver pizzas via bike.  You want to guarantee delivery anywhere on campus in 15 minutes or less; in order to maximize the amount of your tips, you want to deliver as many pizzas as you can and deliver
them as fast as you can.  As a first-year, however, you're still figuring out your way around campus.  You've quickly realized that, because of pedestrian traffic, lights, obstacles, and  placement of bike stands  it is not always best to take the route that covers the shortest distance.  Instead, you would like to know the
quickest way of getting from your dorm room to various parts of campus.

To help you in finding the quickest routes, you map the travel time between some of the most-happening locations on campus.  Below is an attached map showing the key points on campus. 

In [ ]:
Image("images/campus_map.png", width=500)

Let's load two dataframes with the data for each node and edge of this graph respectively.

In [ ]:
nodes = pd.read_csv('data/cornell_pizza_nodes.csv', index_col=0)
display(nodes)
edges = pd.read_csv('data/cornell_pizza_edges.csv', index_col=0)
display(edges.head())

Now, let's pass these to `create_network` and create a graph which we can visualize.

In [ ]:
G = vl.create_network(nodes, edges)
show(vl.tree_plot(G, tree=[], width=500, height=500, show_cost=False))

**Q1:** Your company is based out of Jameson Hall, which is node 1 on the map. What is the index of this node in the graph above? (Hint: Hover over nodes to see their index).

**A:**

_Write your answer here._

Define the source `s` to be Jameson Hall.

In [ ]:
# TODO: Uncomment and set s to the index which corresponds to Jameson Hall
# s = XXX



**Q2:** What is the node that is closest to Jameson Hall (not considering Jameson Hall itself)? In case of ties, choose any of them. Can there be a quicker route to get to this destination (call it point X) than just to go directly there from Jameson Hall? Why or why not?

**A:**

_Write your answer here._

**Q3:** Explain why the predecessor (“`prev(X)`” from class) of `X` must be 0 (Jameson Hall).

**A:**

_Write your answer here._

Run the cell below to generate an interactive plot running Dijkstra's algorithm. If you answered **Q1** and set `s` correctly, the node corresponding to Jameson hall will be red.

After running the cell, click node 0 (in red) and then node 10 (High Rise 5)

In [ ]:
show(vl.assisted_dijkstras_plot(G, s=s, width=500, height=500, show_cost=False))

After clicking those two nodes, you should notice that several things happen:

- Node 0 and 10 turn solid dark blue
- The current shortest path tree is shown by highlighting edges in dark grey
- The table on the bottom updates, and you can see the current version of the Dijkstra table 
- In the current table, node 10 is now starred and prev(10) is marked as node 0

**Q4:** The graph shows three styles of nodes (solid dark blue, red, and light blue). What does each style of node represent?

**A:**

_Write your answer here._

How do we find the next edge to add to the tree (i.e., the next edge to “darken”)? One way is as follows. Compute the travel times for all routes that either go directly from Jameson Hall (0) to a destination or go from Jameson Hall (0) to High Rise 5 (10) and then directly to a destination. Take the shortest of all these routes and add it to the tree. For instance, in this case, we consider the following routes. Node IDs are in parentheses.


| Route | Travel Time |
|-------|-------------|
|Jameson Hall (0) - Clara Dickson Hall (1) | 2*
|Jameson Hall (0) - Helen Newman (11) | 3
|Jameson Hall (0) - Lynah Hockey Rink (13) | 17
|Jameson Hall (0) - High Rise 5 (10) - Holland International Living Center (9) | 3
|Jameson Hall (0) - High Rise 5 (10) - Helen Newman (11) | 7

**Q5:** The routes listed above give current tentative routes to nodes 1, 9, 11, and 13. What is true about those nodes, and only those nodes, at this point in Dijkstra's algorithm?

**A:**

_Write your answer here._

**Q6:** The list above includes two possible routes to node 11: a direct route from Jameson Hall (0) to Helen Newman (11), and a route that goes through High Rise 5 (10). Look at the row for node 11 in the Dijkstra table. Which of these two routes is the table recording as the current best route from Jameson Hall to node 11? How can you tell using only the information shown in the table, without looking back at the graph?

**A:**

_Write your answer here._

Among the tentative routes in the table, the route to Clara Dickson Hall (1) has the shortest travel time. We have marked this route with a `*` to indicate that Clara Dickson Hall is the next node Dijkstra's algorithm is ready to add to the shortest path tree.

**Q7:** Can there be a shorter route to Clara Dickson (1) than the one that we just added? Why or why not? Answer this question in as much generality as possible. (Hint: suppose that there were some shorter path that went from Jameson (0) – SOMEWHERE ELSE – Clara Dickson (1). How would the length of the first edge in this path have to compare to the length of the edge directly between Jameson (0) and Clara Dickson (1)?)

**A:**

_Write your answer here._

Now that Clara Dickson Hall (1) has been added to the shortest path tree, we update the table. First, if two routes lead to the same destination, keep only the shorter route. For example, the route Jameson Hall (0) - High Rise 5 (10) - Helen Newman (11) is longer than the direct route Jameson Hall (0) - Helen Newman (11), so we drop it.

Next, look at the nodes adjacent to the newly added node, Clara Dickson Hall (1). For each one, compute the route that follows the current shortest path from Jameson Hall to Clara Dickson Hall, and then takes one more edge from Clara Dickson Hall to that destination. Add any new routes to the table, and replace old routes if the new route is shorter.

**Q8:** Fill in the missing entries in the table below. Then, as before, cross out any redundant paths (like the Jameson Hall (0) - High Rise 5 (10) - Helen Newman (11) path which we previously dropped).

**A:**

| Route | Travel Time |
|-------|-------------|
|Jameson Hall (0) - Helen Newman (11) | 3
|Jameson Hall (0) - Lynah Hockey Rink (13) | 17
|Jameson Hall (0) - High Rise 5 (10) - Holland International Living Center (9) | 3
|Jameson Hall (0) - Clara Dickson Hall (1) - ? | ?
|Jameson Hall (0) - Clara Dickson Hall (1) - ? | ?
|Jameson Hall (0) - Clara Dickson Hall (1) - ? | ?

On the graph, click Clara Dickson Hall (1). Use the updated table to verify your work. Notice that we discovered a shorter path to Lynah Hockey Rink through Clara Dickson and we see the highlighted edges change accordingly! 

**Q9:** Which node should be clicked next? Call it `X`. What is prev(X) and the shortest path from  Jameson Hall to X?

**A:**

_Write your answer here._

Dijkstra’s Algorithm continues in this manner. In the next step, we compute the travel times to destinations not already marked (i.e. starred/solid dark blue) that are adjacent to the most recently marked node, using routes that involve only already marked nodes as intermediate steps.

**Q10:** Do the next iteration of the algorithm by hand based on your previous answer: write down the nodes that get marked and/or have their labels updated in this iteration.

**A:**

_Write your answer here._

Finish executing Dijkstra's algorithm by clicking the next node to mark at each step. Feel free to use the instance below to avoid scrolling.

In [ ]:
show(vl.assisted_dijkstras_plot(G, s=s, width=500, height=500, show_cost=False))

**Q11:** How can you read out the shortest path from Jameson Hall to other places on campus from the shortest path tree?

**A:**

_Write your answer here._

**Q12:** What is the shortest path (both the route and length) from Jameson Hall (0) to Rhodes Hall (7)?

**A:**

_Write your answer here._

**Q13:** Is the 15 minute guarantee a good one? What about a 30 minute guarantee, assuming some reasonably small fluctuations in the actual travel time due to traffic, etc.

**A:**

_Write your answer here._

**Q14:** Is the shortest path always unique? If yes, why? If no, can you give an example of two nodes in the graph for this lab that have more than one shortest path between them?

**A:**

_Write your answer here._

## Part II: Sensitivity analysis

The next several questions ask you to analyze how sensitive your output is to the precise data that has been used. This is called “performing sensitivity analysis” for the input.

**Q15:** Suppose that there was a parade on campus so that the time to go from Holland International (9) to Cornell High Energy (8) directly was increased to 20 minutes.
Would this change the shortest path tree? How would the shortest path tree change?

**A:**

_Write your answer here._

**Q16:** Can such an increase affect any of the routes that did not originally include the leg from Holland International (9) to Cornell High Energy (8)? Why or why not?

**A:**

_Write your answer here._

Run the cell below to change the weight and generate a plot running Dijkstra's algorithm. Press the next button to see each iteration until all nodes are marked.

In [ ]:
G[9][8]['weight'] = 20  # change weight
show(vl.dijkstras_plot(G, s=s, width=500, height=500, show_cost=False))

**Q17:** Did you make any mistakes in **Q15** and **Q16**?

**A:**

_Write your answer here._

**Q18:** Suppose the travel time from node (9) to node (8) increased from 14 minutes, but not all the way to 20 minutes. By how much could the travel time increase without the shortest path route to node (8) changing?

**A:**

_Write your answer here._

Suppose that your competitor, Better-Than-Adequate Pizza, decides to promote their brand by installing a moving walkway that takes students from Clara Dickson on North Campus to the Lynah hockey rink. This has the effect of reducing the travel time from Clara Dickson (1) to Lynah (13) so that it only takes 5 minutes.

**Q19:** Would this change or alter the shortest path tree? Does this affect any of the routes that did not originally include the leg from Clara Dickson to Lynah? Why or why not?

**A:**

_Write your answer here._

In [ ]:
G[9][8]['weight'] = 14  # change this weight back to original
G[1][13]['weight'] = 5  # change this weight
show(vl.dijkstras_plot(G, s=s, width=500, height=500, show_cost=False))

**Q20:** Suppose that all the travel times obeyed the following inequality:

```text
travel time from A to C <= travel time from A to B + travel time from B to C
```

for all nodes A, B and C. This is called the triangle inequality. Suppose our input was the complete graph (that is, there is an edge between every pair of nodes). If the travel times on this input obeyed the triangle inequality, what would the shortest path tree look like?

**A:**

_Write your answer here._

## Part III: Making it Big - Delivering Pizza in NYC

In this part of the lab, we solve a larger instance of the pizza delivery problem that will otherwise be extremely time-consuming to compute by hand. 

Your pizza delivery service has enjoyed much success and opens up a new shop based at Cornell Tech. You are told that the central location on Roosevelt Island makes it possible to deliver pizzas anywhere in the city within 40 minutes. Your task is to find the best driving routes and to decide whether the 40-minute guarantee is realistic.

Here you will use the actual NYC road network. In this network, a node represents any intersection; edges are road segments that connect intersections. Most streets in New York City are included. Approximate travel times are estimated from millions of Yellow Cab travel times.  

Begin by loading the data files `nyc_nodes.csv` and `nyc_links.csv` from the `data` folder. (Data originally from: https://lab-work.github.io/data/). The data is kept in pandas dataframes. To view the data as tables, run the cells below.

In [ ]:
# load nodes
data = pd.read_csv('data/nyc_nodes.csv')
dfn = pd.DataFrame(data)
# load edges
data = pd.read_csv('data/nyc_links.csv')
dfl = pd.DataFrame(data)

print('Loaded %d nodes and %d edges.' % (dfn.shape[0], dfl.shape[0]))

Use `dfl.head()` to inspect the link data. Note that some streets have multiple edges. This is because there are multiple road segments on some streets. Also included are two delay columns: one for NYC at 8 pm, another at 5pm.

In [ ]:
dfl.head()

We will restrict our focus to a handful of nodes that we treat as Points of Interest (`PoIs`). Our goal is to decide if pizza can be delivered to these locations in a timely fashion.

In [ ]:
# define points of interest (poi)
poi = list((1241986499, 42446461, 42439861,
            103864622, 42428391, 599270647,
            42466966, 42487873))
origin = poi[0] # Roosevelt Island

# define results dataframe
results = pd.DataFrame({'node_id':poi})

Run the cell below to plot the road network and `PoIs`. All nodes except the Points of Interest (`PoIs`) have been made invisible to keep clutter at a minimum.

**Q21:** Do you recognize the Points of Interest (`PoIs`)?

**A:**

_Write your answer here._

In [ ]:
from graph_tools import plotNetwork
plotNetwork(dfn, dfl, title="NYC road network", targets=poi, on_map=True)

Next, load the data into a networkx model and solve. Networkx is a library for dealing with graphs and graph algorithms in Python. Here we use one of networkx's built-in shortest path solvers, but later in the course, we will write our own.

Recall that edges (see `dfl`) were defined by the `start` node and the `end` node. We load the data as a graph in the next cell by specifying (1) that our data sits in `dfl`, (2) that edges start at nodes from the `start` column, (3) that edges end in nodes in the `end` column, and (4) that edge costs are in the cost column. The format is:

```python
G = nx.from_pandas_edgelist(dataframe_of_edges, start_column, end_column, cost_column)
```

**Q22:** Most pizza is delivered around 8pm, so use `delay8pm` as the costs. Explore the travel times to the various Points of Interest. Is a 30-minute guarantee reasonable?

**A:**

_Write your answer here._

In [ ]:
# load networkx model from edge dataset
G = nx.from_pandas_edgelist(dfl, 'start', 'end', ['delay8pm', 'delay5pm'])

In [ ]:
# set delay variable to be 8pm delays
delay = 'delay8pm'

In [ ]:
# solve shortest paths
out = nx.single_source_dijkstra(G, origin, weight=delay)
# record output times
results[delay] = results['node_id'].map(out[0]) / 60

In [ ]:
# inspect the output
results

**Q23:** Next, plot the shortest path tree. Because there are so many nodes, we're only interested in plotting the shortest paths to the  `PoIs`. What do you see about the paths? Are there edges (roads / bridges / driveways) that the shortest paths seem to rely heavily on?

**A:**

_Write your answer here._

In [ ]:
from graph_tools import plotShortestPathTree
plotShortestPathTree(dfn, dfl, out, poi)

**Q24:** As you may have noticed, all deliveries to the west of Roosevelt Island take the Queensboro Bridge. Aside from Hoboken, it seems feasible to deliver to all `PoIs` in close to 30 minutes. But what if there is a major traffic jam on the Queensboro Bridge? Add 30 minutes to edges corresponding to the Queensboro Bridge and re-solve the model. Print the resulting table and shortest path tree.

**A:**

_Write your answer here._

In [ ]:
# Edges corresponding to the Queensboro Bridge.
# Each pair is: (start node, end node) for one road segment in dfl.
queensboro_main_segments = [
    (2089938148, 2089938144),
    (2089938149, 2089938145),
    (2089938146, 42423579),
    (371037077, 371034137),
    (371034141, 276032821),
]

# Check whether a row of dfl is one of the bridge segments.
# We check both directions because the graph treats roads as undirected.
def is_queensboro_segment(row):
    segment = (row['start'], row['end'])
    reverse_segment = (row['end'], row['start'])
    return (
        segment in queensboro_main_segments
        or reverse_segment in queensboro_main_segments
    )

queensboro = dfl.apply(is_queensboro_segment, axis=1)

# define a new cost variable 'qb_cost'
dfl['qb_cost'] = dfl['delay8pm']

# add 30 minutes (in seconds) to the selected Queensboro Bridge segments
dfl.loc[queensboro, 'qb_cost'] = dfl.loc[queensboro, 'delay8pm'] + 1800


In [ ]:
# load networkx model from edge dataset
G = nx.from_pandas_edgelist(dfl, 'start', 'end', ['delay8pm', 'delay5pm', 'qb_cost'])

# set delay variable to be 8pm delays
delay = 'qb_cost'
# solve shortest paths
out = nx.single_source_dijkstra(G, origin, weight=delay)
# record output times
results[delay] = results['node_id'].map(out[0]) / 60

# inspect the output
results

In [ ]:
# plot new paths
plotShortestPathTree(dfn, dfl, out, poi)

You have found the shortest paths!